#### Setup

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import time

# Inicializa
spark = SparkSession.builder \
    .appName("Shuffle_Broadcast") \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# --- TRUQUE DIDÁTICO ---
# O Spark por padrão faz Broadcast se a tabela for < 10MB.
# Desabilita o Broadcast automático para FORÇAR o Shuffle (SortMergeJoin)
# -1 significa "nunca faça broadcast automático"
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

print("Setup pronto e Otimizações automáticas DESLIGADAS.")

Setup pronto e Otimizações automáticas DESLIGADAS.


#### Carregando os Dados

In [3]:
# Carregando as tabelas
df_vendas = spark.read.parquet("data/vendas_parquet")
df_usuarios = spark.read.parquet("data/usuarios_parquet")

print(f"Vendas: {df_vendas.count()} linhas")
print(f"Usuários: {df_usuarios.count()} linhas")

Vendas: 5000000 linhas
Usuários: 50000 linhas


#### SortMergeJoin / Shuffle

In [4]:
print("--- Executando JOIN com SHUFFLE (SortMergeJoin) ---")
start_time = time.time()

# Join Padrão (Sem Broadcast)
df_join_shuffle = df_vendas.join(df_usuarios, df_vendas.id_venda == df_usuarios.id_user, "inner") \
    .groupBy(df_usuarios.estado) \
    .agg(F.sum("valor").alias("total_valor"))

# Ação para forçar execução
df_join_shuffle.show()

print(f"Tempo Shuffle: {time.time() - start_time:.4f} segundos")

--- Executando JOIN com SHUFFLE (SortMergeJoin) ---
+------+--------------------+
|estado|         total_valor|
+------+--------------------+
|    AM|1.5813186100000001E7|
|    GO|1.5648597970000003E7|
|    SP|       1.588734764E7|
|    RS|1.5839735450000001E7|
|    MG|1.5575831999999998E7|
|    BA|1.5633008720000003E7|
|    PE|1.5312952920000006E7|
|    RJ|1.5785169270000005E7|
+------+--------------------+

Tempo Shuffle: 5.7290 segundos


#### Broadcast Join

In [5]:
from pyspark.sql.functions import broadcast

print("--- Executando JOIN com BROADCAST (Map-Side Join) ---")
start_time = time.time()

# Join Otimizado (Forçando o Broadcast explicitamente)
df_join_broadcast = df_vendas.join(broadcast(df_usuarios), df_vendas.id_venda == df_usuarios.id_user, "inner") \
    .groupBy(df_usuarios.estado) \
    .agg(F.sum("valor").alias("total_valor"))

# Ação
df_join_broadcast.show()

print(f"Tempo Broadcast: {time.time() - start_time:.4f} segundos")

--- Executando JOIN com BROADCAST (Map-Side Join) ---
+------+--------------------+
|estado|         total_valor|
+------+--------------------+
|    AM|1.5813186099999985E7|
|    GO|1.5648597970000014E7|
|    SP|1.5887347639999986E7|
|    RS|1.5839735450000016E7|
|    MG| 1.557583200000002E7|
|    BA|1.5633008719999973E7|
|    PE| 1.531295292000005E7|
|    RJ|1.5785169270000007E7|
+------+--------------------+

Tempo Broadcast: 1.5882 segundos


#### Comparação entre joins

In [6]:
print("=== Plano do Shuffle (Lento) ===")
df_join_shuffle.explain()

print("\n=== Plano do Broadcast (Rápido) ===")
df_join_broadcast.explain()

=== Plano do Shuffle (Lento) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[estado#12], functions=[sum(valor#3)])
   +- Exchange hashpartitioning(estado#12, 200), ENSURE_REQUIREMENTS, [plan_id=502]
      +- HashAggregate(keys=[estado#12], functions=[partial_sum(valor#3)])
         +- Project [valor#3, estado#12]
            +- SortMergeJoin [id_venda#0], [id_user#10], Inner
               :- Sort [id_venda#0 ASC NULLS FIRST], false, 0
               :  +- Exchange hashpartitioning(id_venda#0, 200), ENSURE_REQUIREMENTS, [plan_id=494]
               :     +- Filter isnotnull(id_venda#0)
               :        +- FileScan parquet [id_venda#0,valor#3] Batched: true, DataFilters: [isnotnull(id_venda#0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/vendas_parquet], PartitionFilters: [], PushedFilters: [IsNotNull(id_venda)], ReadSchema: struct<id_venda:int,valor:double>
               +- Sort [id_user#10 ASC NULLS FIRS